<a href="https://colab.research.google.com/github/safestepassistant/Python/blob/main/SparkTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Code required to get Spark environment

In [1]:
!pip install pyspark

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.types import *

In [13]:
spark = SparkSession.builder \
    .appName("LegalDocumentFilter") \
    .getOrCreate()

In [14]:
from google.colab import files
uploaded = files.upload()

Saving data-engineer-interview-data (2).csv to data-engineer-interview-data (2) (2).csv


In [17]:
df = spark.read.csv("data-engineer-interview-data (2).csv", header=True, inferSchema=True)
df.show(5)

+--------------+------------+------------+----------------------+--------------------+------------+
|DocumentNumber|DocumentDate|DocumentType|RefersToDocumentNumber|RefersToDocumentYear|     Remarks|
+--------------+------------+------------+----------------------+--------------------+------------+
|            27|   8/16/2021|           C|                  NULL|                NULL|        NULL|
|             1|   2/15/2021|           T|                     8|                2020|2020 0008-00|
|            67|   10/9/2020|           A|                  NULL|                NULL|        NULL|
|           157|   10/9/2020|           A|                  NULL|                NULL|        NULL|
|           189|   10/9/2020|           J|                  NULL|                NULL|        NULL|
+--------------+------------+------------+----------------------+--------------------+------------+
only showing top 5 rows


In [18]:
ref_docs = df.select(
    col("DocumentNumber").alias("RefDocumentNumber"),
    col("DocumentType").alias("RefDocumentType")
)

In [19]:
df_with_refs = df.join(
    ref_docs,
    df["DocumentNumber"] == col("RefDocumentNumber"),
    "left"
)

In [20]:
df_filtered = df_with_refs.withColumn(
    "ToRemove",
    when(
        (col("DocumentType") == "J") & (col("RefDocumentType") == "T"), True
    ).when(
        (col("DocumentType") != "J") & (col("RefDocumentType") == "R"), True
    ).otherwise(False)
)


In [22]:
audit_trail = df_filtered.filter(col("ToRemove") == True).select(
    "DocumentNumber",
    "DocumentType",
    "RefDocumentNumber",
    "RefDocumentType",
    "DocumentDate",
    "Remarks"
)

audit_trail.show(10)

+--------------+------------+-----------------+---------------+------------+-------+
|DocumentNumber|DocumentType|RefDocumentNumber|RefDocumentType|DocumentDate|Remarks|
+--------------+------------+-----------------+---------------+------------+-------+
|           250|           R|              250|              R|   10/9/2020|   NULL|
|            87|           R|               87|              R|    8/7/2020|   NULL|
|           117|           R|              117|              R|   6/21/2019|   NULL|
|            59|           R|               59|              R|  10/15/2018|   NULL|
|            69|           C|               69|              R|  10/15/2018|   NULL|
|            97|           B|               97|              R|  10/27/2017|   NULL|
|            82|           R|               82|              R|  10/17/2016|   NULL|
|           212|           R|              212|              R|  10/17/2016|   NULL|
|            89|           B|               89|              R|  

In [23]:
final_df = df_filtered.filter(col("ToRemove") == False).select(
    "DocumentNumber", "DocumentType", "DocumentDate",
    "RefersToDocumentNumber", "RefersToDocumentYear", "Remarks"
)
final_df.show(10)

+--------------+------------+------------+----------------------+--------------------+------------+
|DocumentNumber|DocumentType|DocumentDate|RefersToDocumentNumber|RefersToDocumentYear|     Remarks|
+--------------+------------+------------+----------------------+--------------------+------------+
|            27|           C|   8/16/2021|                  NULL|                NULL|        NULL|
|             1|           T|   2/15/2021|                     8|                2020|2020 0008-00|
|             1|           T|   2/15/2021|                     8|                2020|2020 0008-00|
|             1|           T|   2/15/2021|                     8|                2020|2020 0008-00|
|            67|           A|   10/9/2020|                  NULL|                NULL|        NULL|
|           157|           A|   10/9/2020|                  NULL|                NULL|        NULL|
|           189|           J|   10/9/2020|                  NULL|                NULL|        NULL|


In [24]:
audit_trail.write.csv("audit_trail.csv", header=True, mode="overwrite")
final_df.write.csv("final_documents.csv", header=True, mode="overwrite")

print("Audit trail and filtered documents saved as CSVs.")

Audit trail and filtered documents saved as CSVs.


In [35]:
print("Original count:", df.count())
print("Removed count:", audit_trail.count())
print("Final count:", final_df.count())

Original count: 146
Removed count: 4
Final count: 142


# Task
Fix the data processing logic in the PySpark notebook using the data from "data-engineer-interview-data (2).csv" to correctly identify and remove documents based on references. Specifically:
1. **Fix Join Logic**: Update the join between the main DataFrame and the reference documents. The current logic incorrectly matches a document to itself; it should instead match a `DocumentNumber` to the `RefersToDocumentNumber` of other records to identify if a 'Termination' (T) or 'Release' (R) exists for it.
2. **Refine Filter Conditions**: Update the `ToRemove` logic so that 'J' type documents are marked for removal only when referred to by a 'T' type document, and all other document types are marked for removal when referred to by an 'R' type document.
3. **Handle Data Integrity**: Resolve the row inflation issue (where the final count exceeds the input count) by ensuring the join does not create a Cartesian product and handles duplicate references correctly.
4. **Validation**: Ensure that the 'Final count' plus the 'Removed count' matches the original input row count (146).

## Fix Join Logic

### Subtask:
Update the Spark join logic to identify relationships between documents based on references rather than matching a document to itself.


**Reasoning**:
I will update the join logic as instructed, filtering for 'T' and 'R' document types, renaming columns to avoid ambiguity, and ensuring distinct records to prevent row inflation when joining with the original document IDs.



In [32]:
from pyspark.sql.functions import col

# 1. Create filtered and renamed reference DataFrame
ref_docs_fixed = df.filter(col("DocumentType").isin("T", "R")) \
    .select(
        col("RefersToDocumentNumber").alias("LinkedDocID"),
        col("DocumentType").alias("ReferencingType")
    ) \
    .filter(col("LinkedDocID").isNotNull()) \
    .distinct()

# 2 & 3. Join original df with ref_docs where DocumentNumber matches the LinkedDocID
df_joined_fixed = df.join(
    ref_docs_fixed,
    df["DocumentNumber"] == ref_docs_fixed["LinkedDocID"],
    "left"
)

# Display sample to verify join logic
print("Fixed Join Result:")
df_joined_fixed.select("DocumentNumber", "DocumentType", "LinkedDocID", "ReferencingType").show(10)
print("Joined row count:", df_joined_fixed.count())

Fixed Join Result:
+--------------+------------+-----------+---------------+
|DocumentNumber|DocumentType|LinkedDocID|ReferencingType|
+--------------+------------+-----------+---------------+
|            27|           C|       NULL|           NULL|
|             1|           T|          1|              R|
|            67|           A|       NULL|           NULL|
|           157|           A|       NULL|           NULL|
|           189|           J|       NULL|           NULL|
|           250|           R|       NULL|           NULL|
|            87|           R|       NULL|           NULL|
|             8|           B|          8|              T|
|             2|           C|       NULL|           NULL|
|            34|           A|       NULL|           NULL|
+--------------+------------+-----------+---------------+
only showing top 10 rows
Joined row count: 146


## Update Filter Conditions

### Subtask:
Refine the logic to mark documents for removal based on the specific rules for 'J' types and other document types when referred to by 'T' or 'R' types.


**Reasoning**:
I will update the 'ToRemove' column logic using the specified conditions for 'J' types and other document types, then create the audit trail and final DataFrames and verify the record counts.



In [33]:
from pyspark.sql.functions import col, when

# 1, 2, 3, 4. Define ToRemove logic based on specific document type rules
df_processed = df_joined_fixed.withColumn(
    "ToRemove",
    when((col("DocumentType") == "J") & (col("ReferencingType") == "T"), True)
    .when((col("DocumentType") != "J") & (col("ReferencingType") == "R"), True)
    .otherwise(False)
)

# 5. Create audit_trail and final_df
audit_trail = df_processed.filter(col("ToRemove") == True)
final_df = df_processed.filter(col("ToRemove") == False)

# 6. Print counts and verify sum
original_count = df.count()
audit_count = audit_trail.count()
final_count = final_df.count()

print(f"Original Count: {original_count}")
print(f"Audit Trail (Removed) Count: {audit_count}")
print(f"Final (Remaining) Count: {final_count}")
print(f"Sum (Audit + Final): {audit_count + final_count}")

if (audit_count + final_count) == original_count:
    print("Verification Successful: Counts match.")
else:
    print("Verification Failed: Counts do not match.")

Original Count: 146
Audit Trail (Removed) Count: 4
Final (Remaining) Count: 142
Sum (Audit + Final): 146
Verification Successful: Counts match.


## Final Task

### Subtask:
Summarize the changes made to the logic and verify that the 'Final count' plus 'Removed count' aligns correctly with the input data.


## Summary:

### Q&A

**Did the final counts align with the input data after fixing the logic?**
Yes. The original input count was 146. After the fix, the 'Removed count' was 4 and the 'Final count' was 142. The sum (142 + 4) equals exactly 146, successfully verifying data integrity.

**How was the row inflation issue resolved?**
Row inflation was resolved by applying the `.distinct()` function to the reference DataFrame (`ref_docs_fixed`) before the join. This ensured that multiple references to the same document did not create duplicate rows (Cartesian products) in the main dataset.

### Data Analysis Key Findings

*   **Join Logic Correction**: The join condition was updated to link `DocumentNumber` from the main table to `RefersToDocumentNumber` (aliased as `LinkedDocID`) from the reference documents.
*   **Targeted Reference Filtering**: The reference set was narrowed down exclusively to 'Termination' (T) and 'Release' (R) document types to act as triggers for removal.
*   **Conditional Removal Rules**:
    *   **'J' type** documents are now only flagged for removal if they are referenced by a **'T' type** document.
    *   **All other document types** are flagged for removal if they are referenced by an **'R' type** document.
*   **Validation Success**: The logic identified 4 records for removal and 142 records for retention, maintaining the total original count of 146.

### Insights or Next Steps

*   **Audit Trail Implementation**: The 4 removed records are stored in a separate `audit_trail` DataFrame, which should be used for future verification or manual review to ensure no critical "J" or other documents were removed in error.
*   **Duplicate Reference Monitoring**: Since `.distinct()` was required to prevent row inflation, it may be beneficial to investigate the source data to understand why multiple 'T' or 'R' documents might refer to the same document number.
